# DM-NeRF / DM-SR — AMP Resume (Kaggle)

Anexe em **Add Input → Notebook Output** a saída do benchmark AMP ou da sessão AMP anterior. Ative GPU e Internet e use **Save Version → Save & Run All**. Não edite código.

Cada sessão treina por até 11 h e salva checkpoint resumível. O alvo atual é 200001 iterações; avaliação e meshing só são executados quando esse alvo for concluído.


In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys

def run(*args):
    subprocess.run([str(x) for x in args], check=True)

run('nvidia-smi')
BASE=Path('/kaggle/temp/dissertacao-dmnerf-amp-resume')
OUTPUT=Path('/kaggle/working/dmnerf-results')
candidates=[]
for state_path in Path('/kaggle/input').rglob('training_state.json'):
    identity_path=state_path.parent/'identity.json'
    ckpt=state_path.parent/'latest.tar'
    if state_path.parent.name!='experiment' or not identity_path.is_file() or not ckpt.is_file():
        continue
    state=json.loads(state_path.read_text())
    identity=json.loads(identity_path.read_text())
    if identity.get('amp') is True:
        candidates.append((int(state['iteration']),state_path.parents[3]))
if not candidates:
    raise RuntimeError('Anexe a saída do benchmark AMP ou da sessão AMP anterior em Add Input → Notebook Output.')
iteration,previous=sorted(candidates,key=lambda item:(item[0],str(item[1])))[-1]
print('Checkpoint selecionado:',previous,'iteração',iteration)
if OUTPUT.exists(): shutil.rmtree(OUTPUT)
shutil.copytree(previous,OUTPUT)
if BASE.exists(): shutil.rmtree(BASE)
BASE.mkdir(parents=True,exist_ok=True)
REPO=BASE/'sdf-generator'; WORK=BASE/'work'
run('git','clone','--branch','dissertacao-2026','--single-branch','https://github.com/Jfsslemos/sdf-generator.git',REPO)
run('git','-C',REPO,'rev-parse','HEAD')
run(sys.executable,REPO/'tools/dmnerf/bootstrap.py','--work',WORK,'--lock',OUTPUT/'environment.freeze.txt')
PYTHON=WORK/'env/bin/python'
run(PYTHON,'-c','import torch; assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0), torch.__version__)')
run(PYTHON,REPO/'tools/dmnerf/prepare.py','--work',WORK)


In [ ]:
run(PYTHON,REPO/'tools/dmnerf/run.py','--work',WORK,'--output',OUTPUT,'--profile','full','--amp')


In [ ]:
state=json.loads((OUTPUT/'raw/study/experiment/training_state.json').read_text())
print(json.dumps(state,indent=2))
remaining=max(0,state['target_steps']-(state['iteration']+1))
print('Iterações restantes:',remaining)
print('Pacote:',OUTPUT.parent/(OUTPUT.name+'-bundle.zip'))


Ao final, envie `dmnerf-results-bundle.zip` ao ChatGPT. Se `complete=false`, crie outra versão usando este mesmo notebook e anexe o output da sessão anterior. Se `complete=true`, a mesma execução já terá tentado avaliação e meshing finais.
